In [1]:
import pandas as pd
import warnings
import numpy as np
from scipy.stats import norm
warnings.filterwarnings("ignore", category=RuntimeWarning) 

combined = pd.read_pickle("/Users/caropark/Desktop/fao_redo/data/combined_update.pkl")
combined

,year,fao_idx,cropname,yield,csif,country,gridcells,yield_lag,yield_lead,whichlag,name,yield_og,iso_a3
0,2000.0,26.0,Cassava,6536.7,0.102158,Dominican Republic,2057,NaN,6976.3,yield,Dominican Republic,6536.7,DOM
1,2001.0,26.0,Cassava,6976.3,0.275784,Dominican Republic,18180,6536.7,6992.4,yield,Dominican Republic,6976.3,DOM
2,2002.0,26.0,Cassava,6992.4,0.285452,Dominican Republic,18183,6976.3,7311.0,yield,Dominican Republic,6992.4,DOM
3,2003.0,26.0,Cassava,7311.0,0.292894,Dominican Republic,18182,6992.4,6550.5,yield,Dominican Republic,7311.0,DOM
4,2004.0,26.0,Cassava,6550.5,0.302970,Dominican Republic,18182,7311.0,6655.9,yield,Dominican Republic,6550.5,DOM
...,...,...,...,...,...,...,...,...,...,...,...,...,...
42235,2019.0,158.0,Wheat,1522.0,0.286319,Bolivia (Plurinational State of),243718,1511.7,1522.0,yield_lead,Bolivia (Plurinational State of) (lead),1199.3,BOL
42236,2020.0,158.0,Wheat,1675.2,0.277672,Bolivia (Plurinational State of),244098,1199.3,1675.2,yield_lead,Bolivia (Plurinational State of) (lead),1522.0,BOL
42237,2021.0,158.0,Wheat,1445.5,0.274657,Bolivia (Plurinational State of),243017,1522.0,1445.5,yield_lead,Bolivia (Plurinational State of) (lead),1675.2,BOL
42238,2022.0,158.0,Wheat,1267.2,0.279872,Bolivia (Plurinational State of),243425,1675.2,1267.2,yield_lead,Bolivia (Plurinational State of) (lead),1445.5,BOL


In [2]:
corrs = combined.groupby(['country', 'cropname'])[['yield_og', 'yield_lag', 'yield_lead']].corrwith(combined['csif'])
corrs['argmax'] = corrs.idxmax(axis=1)
corrs['n'] = combined.groupby(['country', 'cropname']).count()['yield']
corrs = corrs.dropna()
corrs = corrs[corrs['n']>3]

In [3]:
df= corrs[corrs['argmax']!="yield_og"]

# Fisher's Z transformation
def fisher_z(r):
    return 0.5 * np.log((1 + r) / (1 - r))

# p-value calculation for comparing two correlations
def compare_correlations(r1, r2, n):
    z1 = fisher_z(r1)
    z2 = fisher_z(r2)
    diff = z1 - z2
    standard_error = np.sqrt(2 / (n - 3))  # Standard error for the difference
    z_stat = diff / standard_error
    p_value = 2 * norm.sf(abs(z_stat))  # Two-tailed p-value
    return p_value

# Apply the test to the DataFrame
def test_significance(row):
    r1 = row['yield_og']  # yield_og correlation
    r2 = row[row['argmax']]  # correlation of argmax
    n = row['n']
    p_value = compare_correlations(r1, r2, n)
    return p_value
    
# Add significance test results to the DataFrame
df = df.assign(p_value=df.apply(test_significance, axis=1))



In [4]:
print('total obs where lead/lags are better:', len(df),
     '\n', '\n',
     'obs where lead/lags are significantly better at p < 0.1:', len(df[df['p_value']<0.1]), 
     '\n', '\n',
     'percentage:', len(df[df['p_value'] < 0.1 ]) / len(df) * 100,
     '\n', '\n',
     'avg p-value:', df['p_value'].mean() )

total obs where lead/lags are better: 1012 
 
 obs where lead/lags are significantly better at p < 0.1: 33 
 
 percentage: 3.260869565217391 
 
 avg p-value: 0.6413679816874857


In [5]:
supp_table = df[df['p_value']<0.1]
supp_table.columns=  ['corr y', 'corr y-1', 'corr y+1', 'argmax', 'n', 'p_value']
supp_table.to_csv("./plots/lag_sig_update.csv")